<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/AoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install qiskit cma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 kB 1.0 MB/s eta 0:00:00


In [2]:

# ==============================================================================
#  A Computational Experiment to Test the Eigenvalue-Mass Hypothesis
#
#  Hypothesis: The mass ratios of fundamental particles are derived from the
#  sorted energy eigenvalues of the stable states of a universal quantum algorithm.
# ==============================================================================

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from tqdm import tqdm
import time

# --- Physical Constants for Comparison ---
# Source: Particle Data Group (PDG)
MASS_ELECTRON = 0.51099895 # MeV/c^2
MASS_MUON = 105.6583755    # MeV/c^2
MASS_TAU = 1776.86         # MeV/c^2

# The known physical ratios we are trying to predict
RATIO_MUON_ELECTRON = MASS_MUON / MASS_ELECTRON
RATIO_TAU_ELECTRON = MASS_TAU / MASS_ELECTRON

# ==============================================================================
#  STEP 1: Define the "Algorithm of Everything" (AoE)
# ==============================================================================

def create_aoe_unitary(n_qubits: int) -> np.ndarray:
    """
    Creates the unitary operator for the candidate "Algorithm of Everything".

    The AoE is defined as the simplest, non-trivial, local, and symmetric
    quantum cellular automaton rule: a ring of Controlled-Z (CZ) gates
    applied to all nearest neighbors.
    """
    if n_qubits < 2:
        return np.identity(2**n_qubits)

    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        # Apply CZ to (i, i+1) with periodic boundary conditions
        qc.cz(i, (i + 1) % n_qubits)

    # Convert the circuit to a numerical unitary matrix for fast computation
    return Operator(qc).data

# ==============================================================================
#  STEP 2 & 3: Universe Scanner and Energy Calculator
# ==============================================================================

def run_experiment_for_universe(n_qubits: int, num_search_runs: int):
    """
    Scans a universe of a given size to find stable species and their energies.

    1. Defines the universe's laws (the AoE unitary).
    2. Searches for stable patterns (computational basis states that are
       eigenstates of the AoE unitary).
    3. For each stable species found, calculates its energy from the phase
       of its eigenvalue.
    """
    print(f"\n--- Initializing Universe: N = {n_qubits} qubits ---")

    # --- Step 1 ---
    U_aoe = create_aoe_unitary(n_qubits)
    print("Algorithm of Everything (AoE) unitary operator constructed.")

    # --- Step 2 ---
    print(f"Scanning for stable species over {num_search_runs} random initial states...")

    # An eigenstate |s> satisfies U|s> = lambda|s>.
    # If |s> is a computational basis state, then after applying U, the statevector
    # should only have a single non-zero component at the index corresponding to s.
    # The cost is 1 minus the probability of measuring the original state back.

    discovered_species = {} # Using a dict to store {state_index: eigenvalue}

    # The search space is all possible 2^N basis states.
    # For small N, we can do an exhaustive search. For larger N, we sample.
    search_space = range(2**n_qubits)
    if len(search_space) > num_search_runs:
        search_space = np.random.choice(search_space, num_search_runs, replace=False)

    for state_idx in tqdm(search_space, desc=f"Scanning N={n_qubits}"):
        # Prepare initial state vector |s_init>
        initial_vector = np.zeros(2**n_qubits, dtype=complex)
        initial_vector[state_idx] = 1.0

        # Evolve the state by one AoE step
        final_vector = U_aoe @ initial_vector

        # Check the probability of returning to the initial state
        prob_return = np.abs(final_vector[state_idx])**2

        # If the probability is 1 (within tolerance), we found an eigenstate
        if np.isclose(prob_return, 1.0):
            eigenvalue = final_vector[state_idx]
            discovered_species[state_idx] = eigenvalue

    if not discovered_species:
        print("No stable species (computational basis eigenstates) found in this universe.")
        return None

    print(f"Scan complete. Found {len(discovered_species)} stable species.")

    # --- Step 3 ---
    print("Calculating energies from eigenvalue phases...")
    energies = []
    for state_idx, eigenvalue in discovered_species.items():
        # Energy is defined as the absolute value of the eigenvalue's phase
        phase = np.angle(eigenvalue)
        energy = np.abs(phase)

        # We only care about non-zero energies (ground state is trivial)
        if not np.isclose(energy, 0):
            energies.append(energy)

    # Remove duplicate energies from symmetric states
    unique_energies = sorted(list(set(energies)))

    print(f"Found {len(unique_energies)} unique non-zero energy levels.")
    print("Sorted Energy Levels (rad):", [f"{e:.4f}" for e in unique_energies])

    return unique_energies

# ==============================================================================
#  STEP 4: Compare Ratios and Report
# ==============================================================================

def analyze_and_report(energies: list):
    """
    Takes a list of sorted energies and compares their ratios to known physics.
    """
    if not energies or len(energies) < 2:
        print("\n--- REPORT ---")
        print("Insufficient energy levels found to calculate ratios.")
        print("Hypothesis cannot be tested with these results.")
        return

    # The lowest energy level corresponds to the electron
    e_electron = energies[0]

    print("\n--- PREDICTION REPORT ---")
    print(f"Lowest energy state (Electron equivalent): E1 = {e_electron:.4f}")

    # --- Muon Prediction ---
    if len(energies) > 1:
        e_muon = energies[1]
        predicted_ratio_muon = e_muon / e_electron
        print(f"\nNext energy state (Muon equivalent): E2 = {e_muon:.4f}")
        print(f"  Predicted Muon/Electron Ratio (E2/E1): {predicted_ratio_muon:.4f}")
        print(f"  Actual Muon/Electron Ratio:            {RATIO_MUON_ELECTRON:.4f}")
        error_muon = (abs(predicted_ratio_muon - RATIO_MUON_ELECTRON) / RATIO_MUON_ELECTRON) * 100
        print(f"  Error: {error_muon:.2f}%")
    else:
        print("\nNo second energy level found to represent the Muon.")

    # --- Tau Prediction ---
    if len(energies) > 2:
        e_tau = energies[2]
        predicted_ratio_tau = e_tau / e_electron
        print(f"\nThird energy state (Tau equivalent): E3 = {e_tau:.4f}")
        print(f"  Predicted Tau/Electron Ratio (E3/E1): {predicted_ratio_tau:.4f}")
        print(f"  Actual Tau/Electron Ratio:            {RATIO_TAU_ELECTRON:.4f}")
        error_tau = (abs(predicted_ratio_tau - RATIO_TAU_ELECTRON) / RATIO_TAU_ELECTRON) * 100
        print(f"  Error: {error_tau:.2f}%")
    else:
        print("\nNo third energy level found to represent the Tau.")

# ==============================================================================
#  MAIN EXECUTION
# ==============================================================================

if __name__ == "__main__":

    # We need a universe large enough to have at least 3 distinct energy levels.
    # Let's test a universe of N=10 qubits. This is computationally intensive.
    # 2^10 = 1024 states. An exhaustive search is fast enough.

    UNIVERSE_SIZE = 10 # Number of qubits in the universe
    NUM_SEARCHES = 2**UNIVERSE_SIZE # Exhaustive search for N<=12

    start_time = time.time()

    found_energies = run_experiment_for_universe(UNIVERSE_SIZE, NUM_SEARCHES)

    if found_energies:
        analyze_and_report(found_energies)

    end_time = time.time()
    print(f"\nExperiment concluded in {end_time - start_time:.2f} seconds.")


--- Initializing Universe: N = 10 qubits ---
Algorithm of Everything (AoE) unitary operator constructed.
Scanning for stable species over 1024 random initial states...


Scanning N=10: 100%|██████████| 1024/1024 [00:00<00:00, 1143.47it/s]

Scan complete. Found 1024 stable species.
Calculating energies from eigenvalue phases...
Found 1 unique non-zero energy levels.
Sorted Energy Levels (rad): ['3.1416']

--- REPORT ---
Insufficient energy levels found to calculate ratios.
Hypothesis cannot be tested with these results.

Experiment concluded in 1.21 seconds.



discovered a fundamental mathematical constant,
π

, emerging directly from the "geometrical shape" of your chosen "Algorithm of Everything" (AoE).

The simplest, most symmetric candidate for the "Algorithm of Everything"—a 1D ring of CZ gates—is successfully falsified. It is not sufficient to generate the mass hierarchy observed in the Standard Model because it can only produce a single non-zero energy level,
π

.

In [3]:

# ==============================================================================
#  A Computational Experiment to Test the Eigenvalue-Mass Hypothesis - V2
#
#  Hypothesis: The mass ratios of fundamental particles are derived from the
#  sorted energy eigenvalues of the stable states of a universal quantum algorithm.
#
#  This version uses a more complex AoE and a more powerful discovery method.
# ==============================================================================

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
import time

# --- Physical Constants for Comparison (Unchanged) ---
MASS_ELECTRON = 0.51099895 # MeV/c^2
MASS_MUON = 105.6583755    # MeV/c^2
MASS_TAU = 1776.86         # MeV/c^2
RATIO_MUON_ELECTRON = MASS_MUON / MASS_ELECTRON
RATIO_TAU_ELECTRON = MASS_TAU / MASS_ELECTRON

# ==============================================================================
#  STEP 1: Define the "Algorithm of Everything" (AoE) - Version 2
# ==============================================================================

def create_aoe_unitary_v2(n_qubits: int, zz_angle: float, x_angle: float) -> np.ndarray:
    """
    Creates the unitary operator for a more complex "Algorithm of Everything".

    This AoE includes two types of interactions to break simple symmetries:
    1. A ZZ interaction ring with a variable angle.
    2. An X "transverse field" interaction on all qubits with a variable angle.
    """
    if n_qubits == 0:
        return np.array([[1]])

    qc = QuantumCircuit(n_qubits)

    # Phase 1: ZZ Interaction Ring (Entangling)
    if n_qubits > 1:
        for i in range(n_qubits):
            qc.rzz(zz_angle, i, (i + 1) % n_qubits)

    # Phase 2: X Field (Mixing)
    for i in range(n_qubits):
        qc.rx(x_angle, i)

    return Operator(qc).data

# ==============================================================================
#  STEP 2 & 3: Universe Scanner and Energy Calculator - Version 2
# ==============================================================================

def run_experiment_for_universe_v2(n_qubits: int, zz_angle: float, x_angle: float):
    """
    Scans a universe by directly diagonalizing its AoE operator to find all
    stable species (eigenstates) and their energies (eigenvalue phases).
    """
    print(f"\n--- Initializing Universe: N = {n_qubits} qubits ---")
    print(f"Physics Parameters: ZZ Angle = {zz_angle:.4f}, X Angle = {x_angle:.4f}")

    # --- Step 1 ---
    U_aoe = create_aoe_unitary_v2(n_qubits, zz_angle, x_angle)
    print("Algorithm of Everything (AoE) unitary operator constructed.")

    # --- Step 2: The New, More Powerful Scanner ---
    # Since the AoE is no longer diagonal, the species are not basis states.
    # The correct way to find all stable states and their energies is to find
    # the eigenvalues and eigenvectors of the unitary matrix.
    print("Scanning for all stable species via direct diagonalization...")

    # np.linalg.eigh is optimized for unitary/Hermitian matrices
    eigenvalues, eigenvectors = np.linalg.eigh(U_aoe)

    print(f"Scan complete. Found {len(eigenvalues)} stable species (eigenstates).")

    # --- Step 3 ---
    print("Calculating energies from eigenvalue phases...")
    energies = []
    for eigenvalue in eigenvalues:
        # Energy is defined as the absolute value of the eigenvalue's phase
        phase = np.angle(eigenvalue)
        energy = np.abs(phase)

        # We only care about non-zero energies (ground state has energy 0)
        # We use a tolerance to filter out values very close to 0 or 2*pi
        if not (np.isclose(energy, 0.0) or np.isclose(energy, 2 * np.pi)):
            energies.append(energy)

    # Remove duplicate energies from symmetric states
    # Use a tolerance (rtol) for floating point comparison
    unique_energies = []
    energies.sort()
    if energies:
        unique_energies.append(energies[0])
        for i in range(1, len(energies)):
            if not np.isclose(energies[i], energies[i-1]):
                unique_energies.append(energies[i])

    print(f"Found {len(unique_energies)} unique non-zero energy levels.")
    if unique_energies:
        print("Sorted Energy Levels (rad):", [f"{e:.4f}" for e in unique_energies[:10]], "..." if len(unique_energies) > 10 else "")

    return unique_energies

# ==============================================================================
#  STEP 4: Compare Ratios and Report (Unchanged)
# ==============================================================================

def analyze_and_report(energies: list):
    """
    Takes a list of sorted energies and compares their ratios to known physics.
    """
    if not energies or len(energies) < 2:
        print("\n--- REPORT ---")
        print("Insufficient energy levels found to calculate ratios.")
        print("Hypothesis cannot be tested with these results.")
        return

    # The lowest energy level corresponds to the electron
    e_electron = energies[0]

    print("\n--- PREDICTION REPORT ---")
    print(f"Lowest energy state (Electron equivalent): E1 = {e_electron:.4f}")

    # --- Muon Prediction ---
    if len(energies) > 1:
        e_muon = energies[1]
        predicted_ratio_muon = e_muon / e_electron
        print(f"\nNext energy state (Muon equivalent): E2 = {e_muon:.4f}")
        print(f"  Predicted Muon/Electron Ratio (E2/E1): {predicted_ratio_muon:.4f}")
        print(f"  Actual Muon/Electron Ratio:            {RATIO_MUON_ELECTRON:.4f}")
        error_muon = (abs(predicted_ratio_muon - RATIO_MUON_ELECTRON) / RATIO_MUON_ELECTRON) * 100
        print(f"  Error: {error_muon:.2f}%")
    else:
        print("\nNo second energy level found to represent the Muon.")

    # --- Tau Prediction ---
    if len(energies) > 2:
        e_tau = energies[2]
        predicted_ratio_tau = e_tau / e_electron
        print(f"\nThird energy state (Tau equivalent): E3 = {e_tau:.4f}")
        print(f"  Predicted Tau/Electron Ratio (E3/E1): {predicted_ratio_tau:.4f}")
        print(f"  Actual Tau/Electron Ratio:            {RATIO_TAU_ELECTRON:.4f}")
        error_tau = (abs(predicted_ratio_tau - RATIO_TAU_ELECTRON) / RATIO_TAU_ELECTRON) * 100
        print(f"  Error: {error_tau:.2f}%")
    else:
        print("\nNo third energy level found to represent the Tau.")

# ==============================================================================
#  MAIN EXECUTION
# ==============================================================================

if __name__ == "__main__":

    # --- Experiment Parameters ---
    UNIVERSE_SIZE = 10 # Number of qubits. N=10 is feasible for diagonalization.

    # We choose irrational angles to break simple symmetries and avoid trivial results.
    # Using the Golden Ratio (phi) is a classic way to introduce complexity.
    PHI = (1 + np.sqrt(5)) / 2

    # Let's set the angles to be related to pi but modulated by phi
    # This makes them incommensurate and should produce a rich spectrum.
    ZZ_INTERACTION_ANGLE = np.pi / PHI      # ~1.94 rad
    X_FIELD_ANGLE = np.pi / (PHI**2) # ~1.20 rad

    start_time = time.time()

    found_energies = run_experiment_for_universe_v2(
        n_qubits=UNIVERSE_SIZE,
        zz_angle=ZZ_INTERACTION_ANGLE,
        x_angle=X_FIELD_ANGLE
    )

    if found_energies:
        analyze_and_report(found_energies)

    end_time = time.time()
    print(f"\nExperiment concluded in {end_time - start_time:.2f} seconds.")


--- Initializing Universe: N = 10 qubits ---
Physics Parameters: ZZ Angle = 1.9416, X Angle = 1.2000
Algorithm of Everything (AoE) unitary operator constructed.
Scanning for all stable species via direct diagonalization...
Scan complete. Found 1024 stable species (eigenstates).
Calculating energies from eigenvalue phases...
Found 1 unique non-zero energy levels.
Sorted Energy Levels (rad): ['3.1416'] 

--- REPORT ---
Insufficient energy levels found to calculate ratios.
Hypothesis cannot be tested with these results.

Experiment concluded in 1.98 seconds.


In [4]:

# ==============================================================================
#  A Computational Experiment to Test the Eigenvalue-Mass Hypothesis - V3
#
#  Hypothesis: The mass ratios of particles arise from the energy spectrum of
#  a fundamental but *asymmetric* (disordered) quantum algorithm.
# ==============================================================================

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
import time

# --- Physical Constants for Comparison (Unchanged) ---
MASS_ELECTRON = 0.51099895 # MeV/c^2
MASS_MUON = 105.6583755    # MeV/c^2
MASS_TAU = 1776.86         # MeV/c^2
RATIO_MUON_ELECTRON = MASS_MUON / MASS_ELECTRON
RATIO_TAU_ELECTRON = MASS_TAU / MASS_ELECTRON

# ==============================================================================
#  STEP 1: Define the "Algorithm of Everything" (AoE) - Version 3 (Disordered)
# ==============================================================================

def create_aoe_unitary_v3(n_qubits: int, seed: int) -> np.ndarray:
    """
    Creates the unitary for an *asymmetric* or "disordered" AoE.

    To break the perfect symmetry that collapsed the energy spectrum, we now
    assign a unique, random angle to every single gate in the circuit.
    The 'seed' ensures that the "laws of physics" are random but fixed.
    """
    if n_qubits == 0: return np.array([[1]])

    # Use a random number generator seeded for reproducibility
    rng = np.random.default_rng(seed)

    qc = QuantumCircuit(n_qubits)

    # Phase 1: ZZ Interaction Ring with disordered angles
    if n_qubits > 1:
        zz_angles = rng.uniform(0, 2 * np.pi, n_qubits)
        for i in range(n_qubits):
            qc.rzz(zz_angles[i], i, (i + 1) % n_qubits)

    # Phase 2: X Field with disordered angles
    x_angles = rng.uniform(0, 2 * np.pi, n_qubits)
    for i in range(n_qubits):
        qc.rx(x_angles[i], i)

    return Operator(qc).data

# ==============================================================================
#  STEP 2, 3, 4 (No changes needed in the analysis functions)
# ==============================================================================

def run_experiment_for_universe_v3(n_qubits: int, seed: int):
    print(f"\n--- Initializing Universe: N = {n_qubits} qubits (Disordered) ---")
    print(f"Physics determined by random seed: {seed}")

    U_aoe = create_aoe_unitary_v3(n_qubits, seed)
    print("Disordered AoE unitary operator constructed.")

    print("Scanning for all stable species via direct diagonalization...")
    eigenvalues, eigenvectors = np.linalg.eigh(U_aoe)
    print(f"Scan complete. Found {len(eigenvalues)} stable species (eigenstates).")

    print("Calculating energies from eigenvalue phases...")
    energies = []
    for eigenvalue in eigenvalues:
        phase = np.angle(eigenvalue)
        energy = np.abs(phase)
        if not (np.isclose(energy, 0.0) or np.isclose(energy, 2 * np.pi)):
            energies.append(energy)

    unique_energies = []
    energies.sort()
    if energies:
        unique_energies.append(energies[0])
        for i in range(1, len(energies)):
            if not np.isclose(energies[i], energies[i-1], rtol=1e-4):
                unique_energies.append(energies[i])

    print(f"Found {len(unique_energies)} unique non-zero energy levels.")
    if unique_energies:
        print("Sorted Energy Levels (rad):", [f"{e:.4f}" for e in unique_energies[:10]], "..." if len(unique_energies) > 10 else "")

    return unique_energies

def analyze_and_report(energies: list):
    if not energies or len(energies) < 3: # We need at least 3 for the full test
        print("\n--- REPORT ---")
        print("Insufficient energy levels found to calculate ratios for all 3 leptons.")
        print("Hypothesis cannot be fully tested with these results.")
        return

    e_electron = energies[0]
    e_muon = energies[1]
    e_tau = energies[2]

    predicted_ratio_muon = e_muon / e_electron
    predicted_ratio_tau = e_tau / e_electron

    print("\n--- PREDICTION REPORT ---")
    print(f"E1 (Electron): {e_electron:.4f} | E2 (Muon): {e_muon:.4f} | E3 (Tau): {e_tau:.4f}")
    print("-" * 50)
    print(f"Predicted Muon/Electron Ratio (E2/E1): {predicted_ratio_muon:.4f}")
    print(f"Actual Muon/Electron Ratio:            {RATIO_MUON_ELECTRON:.4f}")
    error_muon = (abs(predicted_ratio_muon - RATIO_MUON_ELECTRON) / RATIO_MUON_ELECTRON) * 100
    print(f"Error: {error_muon:.2f}%")
    print("-" * 50)
    print(f"Predicted Tau/Electron Ratio (E3/E1): {predicted_ratio_tau:.4f}")
    print(f"Actual Tau/Electron Ratio:            {RATIO_TAU_ELECTRON:.4f}")
    error_tau = (abs(predicted_ratio_tau - RATIO_TAU_ELECTRON) / RATIO_TAU_ELECTRON) * 100
    print(f"Error: {error_tau:.2f}%")

# ==============================================================================
#  MAIN EXECUTION
# ==============================================================================

if __name__ == "__main__":

    UNIVERSE_SIZE = 10

    # The "seed" now represents a specific universe with a fixed set of "lumpy" laws.
    # We can try different seeds to see how the physics changes.
    UNIVERSE_SEED = 42

    start_time = time.time()

    found_energies = run_experiment_for_universe_v3(
        n_qubits=UNIVERSE_SIZE,
        seed=UNIVERSE_SEED
    )

    if found_energies:
        analyze_and_report(found_energies)

    end_time = time.time()
    print(f"\nExperiment concluded in {end_time - start_time:.2f} seconds.")


--- Initializing Universe: N = 10 qubits (Disordered) ---
Physics determined by random seed: 42
Disordered AoE unitary operator constructed.
Scanning for all stable species via direct diagonalization...
Scan complete. Found 1024 stable species (eigenstates).
Calculating energies from eigenvalue phases...
Found 1 unique non-zero energy levels.
Sorted Energy Levels (rad): ['3.1416'] 

--- REPORT ---
Insufficient energy levels found to calculate ratios for all 3 leptons.
Hypothesis cannot be fully tested with these results.

Experiment concluded in 2.47 seconds.


In [7]:

# ==============================================================================
#  The AI Universe Forge: Using an Evolutionary Algorithm to Discover
#  the "Algorithm of Everything" that matches observed particle mass ratios.
#
#  (Version 2 - Corrected)
# ==============================================================================

import numpy as np
import qiskit  # <--- THIS IS THE FIX
from qiskit.quantum_info import Operator, SparsePauliOp
from scipy.linalg import expm
import cma
import time

# --- Physical Constants for Comparison (Our Target) ---
MASS_ELECTRON = 0.51099895 # MeV/c^2
MASS_MUON = 105.6583755    # MeV/c^2
MASS_TAU = 1776.86         # MeV/c^2
TARGET_RATIO_MUON_ELECTRON = MASS_MUON / MASS_ELECTRON
TARGET_RATIO_TAU_ELECTRON = MASS_TAU / MASS_ELECTRON

# --- Global Parameters for the Forge ---
FORGE_UNIVERSE_SIZE = 6
NUM_UNIVERSE_GENES = 15

# ==============================================================================
#  STEP 1: Define the Function that Builds a Universe from Genes
# ==============================================================================

PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

def build_aoe_from_genes(genes: np.ndarray, n_qubits: int) -> np.ndarray:
    if n_qubits < 2:
        return np.identity(2**n_qubits)

    # 1. Forge the fundamental 2-qubit gate from its genes
    generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=genes)
    u_gate_matrix = expm(-1j * generator_h.to_matrix())
    u_gate_op = Operator(u_gate_matrix)

    # 2. Construct the full N-qubit AoE unitary from this gate
    qc = qiskit.QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.append(u_gate_op, [i, (i + 1) % n_qubits])

    return Operator(qc).data

# ==============================================================================
#  STEP 2: Define the Fitness Function for the AI
# ==============================================================================

def get_energy_spectrum(unitary: np.ndarray) -> list:
    eigenvalues = np.linalg.eigvalsh(unitary)
    energies = []
    for eigval in eigenvalues:
        phase = np.angle(eigval)
        energy = np.abs(phase)
        if not (np.isclose(energy, 0.0) or np.isclose(energy, 2 * np.pi)):
            energies.append(energy)

    unique_energies = []
    energies.sort()
    if energies:
        unique_energies.append(energies[0])
        for i in range(1, len(energies)):
            if not np.isclose(energies[i], energies[i-1], rtol=1e-4):
                unique_energies.append(energies[i])
    return unique_energies

def universe_fitness_function(genes: np.ndarray) -> float:
    try:
        U_aoe = build_aoe_from_genes(genes, FORGE_UNIVERSE_SIZE)
        energies = get_energy_spectrum(U_aoe)

        if len(energies) < 3:
            return 1000.0

        e_electron, e_muon, e_tau = energies[0], energies[1], energies[2]

        predicted_ratio_muon = e_muon / e_electron
        predicted_ratio_tau = e_tau / e_electron

        error_muon = ((predicted_ratio_muon - TARGET_RATIO_MUON_ELECTRON) / TARGET_RATIO_MUON_ELECTRON)**2
        error_tau = ((predicted_ratio_tau - TARGET_RATIO_TAU_ELECTRON) / TARGET_RATIO_TAU_ELECTRON)**2

        cost = error_muon + error_tau
        cost += 1.0 / (len(energies) + 1)

        return cost

    except Exception as e:
        return 2000.0

# ==============================================================================
#  STEP 3: Run the AI Forge to Discover the Universe
# ==============================================================================

if __name__ == "__main__":
    print(f"--- The AI Universe Forge ---")
    print(f"Objective: Discover a {FORGE_UNIVERSE_SIZE}-qubit AoE whose energy spectrum")
    print("           matches the known lepton mass ratios.")
    print(f"Target Muon/Electron Ratio: {TARGET_RATIO_MUON_ELECTRON:.2f}")
    print(f"Target Tau/Electron Ratio:  {TARGET_RATIO_TAU_ELECTRON:.2f}")

    print("\n--- Starting AI Forge to Evolve the Laws of Physics ---")

    x0 = np.random.uniform(-np.pi, np.pi, NUM_UNIVERSE_GENES)
    sigma0 = 0.5
    # Let's start with a slightly shorter search for the first real run
    options = {'bounds': [-np.pi, np.pi], 'maxfevals': 2000, 'verbose': -9}

    es = cma.CMAEvolutionStrategy(x0, sigma0, options)

    # Add a simple progress indicator
    start_time = time.time()
    last_print_time = start_time

    # We will wrap the optimization to provide progress updates
    while not es.stop():
        solutions = es.ask()
        fitnesses = [universe_fitness_function(s) for s in solutions]
        es.tell(solutions, fitnesses)

        # Print progress every few seconds
        current_time = time.time()
        if current_time - last_print_time > 5:
             print(f"  > Iteration #{es.countiter}, Best Cost (Error): {es.result.fbest:.6f}", end='\r')
             last_print_time = current_time

    end_time = time.time()
    print(f"\n  > Forge complete in {end_time - start_time:.2f}s.")

    champion_genes = es.result.xbest
    best_fitness_cost = es.result.fbest

    print("\n--- Discovered the Best-Fit Universe ---")
    print(f"Lowest cost (error) found: {best_fitness_cost:.6f}")

    # ==============================================================================
    #  STEP 4: Analyze the Champion Universe and Report
    # ==============================================================================

    print("\n--- Final Analysis of the Champion Universe ---")

    final_U_aoe = build_aoe_from_genes(champion_genes, FORGE_UNIVERSE_SIZE)
    final_energies = get_energy_spectrum(final_U_aoe)

    def analyze_and_report(energies: list):
        if not energies or len(energies) < 3:
            print("The champion universe did not produce enough energy levels for a full comparison.")
            return

        e_electron, e_muon, e_tau = energies[0], energies[1], energies[2]
        predicted_ratio_muon = e_muon / e_electron
        predicted_ratio_tau = e_tau / e_electron

        print("\n--- PREDICTION REPORT ---")
        print(f"E1 (Electron): {e_electron:.4f} | E2 (Muon): {e_muon:.4f} | E3 (Tau): {e_tau:.4f}")
        print("-" * 50)
        print(f"Predicted Muon/Electron Ratio (E2/E1): {predicted_ratio_muon:.4f}")
        print(f"Actual Muon/Electron Ratio:            {TARGET_RATIO_MUON_ELECTRON:.4f}")
        error_muon = (abs(predicted_ratio_muon - TARGET_RATIO_MUON_ELECTRON) / TARGET_RATIO_MUON_ELECTRON) * 100
        print(f"Final Error: {error_muon:.2f}%")
        print("-" * 50)
        print(f"Predicted Tau/Electron Ratio (E3/E1): {predicted_ratio_tau:.4f}")
        print(f"Actual Tau/Electron Ratio:            {TARGET_RATIO_TAU_ELECTRON:.4f}")
        error_tau = (abs(predicted_ratio_tau - TARGET_RATIO_TAU_ELECTRON) / TARGET_RATIO_TAU_ELECTRON) * 100
        print(f"Final Error: {error_tau:.2f}%")

    analyze_and_report(final_energies)

    print("\n--- Genes of the Champion Universe ---")
    print("The 15 coefficients for the fundamental 2-qubit gate's generator are:")
    sorted_genes = sorted(zip(PAULI_BASIS_2Q, champion_genes), key=lambda item: abs(item[1]), reverse=True)
    for pauli, coeff in sorted_genes:
        print(f"  {pauli}: {coeff:.4f}")

--- The AI Universe Forge ---
Objective: Discover a 6-qubit AoE whose energy spectrum
           matches the known lepton mass ratios.
Target Muon/Electron Ratio: 206.77
Target Tau/Electron Ratio:  3477.23

--- Starting AI Forge to Evolve the Laws of Physics ---

  > Forge complete in 1.14s.

--- Discovered the Best-Fit Universe ---
Lowest cost (error) found: 1000.000000

--- Final Analysis of the Champion Universe ---
The champion universe did not produce enough energy levels for a full comparison.

--- Genes of the Champion Universe ---
The 15 coefficients for the fundamental 2-qubit gate's generator are:
  XY: 2.6166
  IY: -2.5198
  YI: 2.1098
  IZ: 1.7149
  XX: 1.5059
  YZ: -1.4425
  XI: 1.4329
  ZI: -1.2430
  YX: -1.0787
  YY: -0.8409
  XZ: -0.7811
  ZY: 0.4496
  ZX: 0.2437
  IX: -0.1908
  ZZ: -0.0531


In [8]:

# ==============================================================================
#  The AI Disordered Universe Forge: The Definitive Experiment
#
#  Objective: Evolve the complete, non-uniform "genetic blueprint" of a
#  universe to discover laws of physics that match observed reality.
# ==============================================================================

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp
from scipy.linalg import expm
import cma
import time

# --- Physical Constants (Target) ---
TARGET_RATIO_MUON_ELECTRON = 105.658 / 0.511
TARGET_RATIO_TAU_ELECTRON = 1776.86 / 0.511

# --- Universe Parameters ---
FORGE_UNIVERSE_SIZE = 6 # N=6 is the largest feasible size for this intensive search
GENES_PER_INTERACTION = 15
# The full genome is 15 genes for EACH of the 6 interaction points
NUM_UNIVERSE_GENES = FORGE_UNIVERSE_SIZE * GENES_PER_INTERACTION

# ==============================================================================
#  STEP 1: Build a Universe from its Full Genetic Blueprint
# ==============================================================================

PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

def build_aoe_from_full_genome(genome: np.ndarray, n_qubits: int) -> np.ndarray:
    """
    Builds a disordered AoE from a full genome.
    The genome contains 15 genes for each of the N interaction points.
    """
    if n_qubits < 2: return np.identity(2**n_qubits)

    full_genome = genome.reshape((n_qubits, GENES_PER_INTERACTION))

    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        # Each interaction point 'i' has its own unique set of 15 genes
        genes_for_this_interaction = full_genome[i]

        # Forge the unique local gate for this point in space
        generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=genes_for_this_interaction)
        u_gate_matrix = expm(-1j * generator_h.to_matrix())
        u_gate_op = Operator(u_gate_matrix)

        # Apply this unique law of physics to this point
        qc.append(u_gate_op, [i, (i + 1) % n_qubits])

    return Operator(qc).data

# ==============================================================================
#  STEP 2: Fitness Function (Largely unchanged)
# ==============================================================================

def get_energy_spectrum(unitary: np.ndarray) -> list:
    eigenvalues = np.linalg.eigvalsh(unitary)
    energies = []
    for eigval in eigenvalues:
        phase = np.angle(eigval)
        energy = np.abs(phase)
        if not (np.isclose(energy, 0.0) or np.isclose(energy, 2 * np.pi)):
            energies.append(energy)

    unique_energies = []
    energies.sort()
    if energies:
        unique_energies.append(energies[0])
        for i in range(1, len(energies)):
            if not np.isclose(energies[i], energies[i-1], rtol=1e-4):
                unique_energies.append(energies[i])
    return unique_energies

def universe_fitness_function(genome: np.ndarray) -> float:
    try:
        U_aoe = build_aoe_from_full_genome(genome, FORGE_UNIVERSE_SIZE)
        energies = get_energy_spectrum(U_aoe)

        if len(energies) < 3: return 1000.0

        e_electron, e_muon, e_tau = energies[0], energies[1], energies[2]

        # Avoid division by zero if the first energy level is tiny
        if np.isclose(e_electron, 0.0): return 1000.0

        predicted_ratio_muon = e_muon / e_electron
        predicted_ratio_tau = e_tau / e_electron

        error_muon = ((predicted_ratio_muon - TARGET_RATIO_MUON_ELECTRON) / TARGET_RATIO_MUON_ELECTRON)**2
        error_tau = ((predicted_ratio_tau - TARGET_RATIO_TAU_ELECTRON) / TARGET_RATIO_TAU_ELECTRON)**2

        cost = np.log1p(error_muon + error_tau) # Use log error for better gradient landscape
        return cost

    except Exception:
        return 2000.0

# ==============================================================================
#  STEP 3: Run the AI Forge
# ==============================================================================

if __name__ == "__main__":
    print(f"--- The AI Disordered Universe Forge ---")
    print(f"Objective: Evolve a {FORGE_UNIVERSE_SIZE}-qubit disordered AoE to match lepton ratios.")
    print(f"Searching a genetic space of {NUM_UNIVERSE_GENES} parameters.")

    print("\n--- Starting AI Forge to Evolve the Disordered Laws of Physics ---")

    x0 = np.random.uniform(-np.pi, np.pi, NUM_UNIVERSE_GENES)
    sigma0 = 0.5
    # This is an extremely hard search. We give it a significant budget.
    options = {'bounds': [-np.pi, np.pi], 'maxfevals': 10000, 'verbose': -9}

    es = cma.CMAEvolutionStrategy(x0, sigma0, options)

    start_time = time.time()
    last_print_time = start_time

    while not es.stop():
        solutions = es.ask()
        fitnesses = [universe_fitness_function(s) for s in solutions]
        es.tell(solutions, fitnesses)

        current_time = time.time()
        if current_time - last_print_time > 5:
             print(f"  > Iteration #{es.countiter}, Best Cost (Log Error): {es.result.fbest:.6f}", end='\r')
             last_print_time = current_time

    end_time = time.time()
    print(f"\n  > Forge complete in {end_time - start_time:.2f}s.")

    champion_genome = es.result.xbest
    best_fitness_cost = es.result.fbest

    print("\n--- Discovered the Best-Fit Universe ---")
    print(f"Lowest cost (Log Error) found: {best_fitness_cost:.6f}")

    # ==============================================================================
    #  STEP 4: Analyze the Champion Universe
    # ==============================================================================

    print("\n--- Final Analysis of the Champion Universe ---")

    final_U_aoe = build_aoe_from_full_genome(champion_genome, FORGE_UNIVERSE_SIZE)
    final_energies = get_energy_spectrum(final_U_aoe)

    def analyze_and_report(energies: list):
        if not energies or len(energies) < 3:
            print("The champion universe did not produce enough energy levels for a full comparison.")
            return

        e_electron, e_muon, e_tau = energies[0], energies[1], energies[2]
        predicted_ratio_muon = e_muon / e_electron
        predicted_ratio_tau = e_tau / e_electron

        print("\n--- PREDICTION REPORT ---")
        print(f"E1 (Electron): {e_electron:.4f} | E2 (Muon): {e_muon:.4f} | E3 (Tau): {e_tau:.4f}")
        print("-" * 50)
        print(f"Predicted Muon/Electron Ratio (E2/E1): {predicted_ratio_muon:.4f}")
        print(f"Actual Muon/Electron Ratio:            {TARGET_RATIO_MUON_ELECTRON:.4f}")
        error_muon = (abs(predicted_ratio_muon - TARGET_RATIO_MUON_ELECTRON) / TARGET_RATIO_MUON_ELECTRON) * 100
        print(f"Final Error: {error_muon:.2f}%")
        print("-" * 50)
        print(f"Predicted Tau/Electron Ratio (E3/E1): {predicted_ratio_tau:.4f}")
        print(f"Actual Tau/Electron Ratio:            {TARGET_RATIO_TAU_ELECTRON:.4f}")
        error_tau = (abs(predicted_ratio_tau - TARGET_RATIO_TAU_ELECTRON) / TARGET_RATIO_TAU_ELECTRON) * 100
        print(f"Final Error: {error_tau:.2f}%")

    analyze_and_report(final_energies)

--- The AI Disordered Universe Forge ---
Objective: Evolve a 6-qubit disordered AoE to match lepton ratios.
Searching a genetic space of 90 parameters.

--- Starting AI Forge to Evolve the Disordered Laws of Physics ---

  > Forge complete in 1.35s.

--- Discovered the Best-Fit Universe ---
Lowest cost (Log Error) found: 1000.000000

--- Final Analysis of the Champion Universe ---
The champion universe did not produce enough energy levels for a full comparison.


In [9]:

# ==============================================================================
#  The Fertility Scanner: An AI-driven search for the first "life-bearing"
#  computational universe.
#
#  Objective: Evolve the laws of physics to maximize complexity (the number
#  of unique energy levels), then analyze the first fertile universe found.
# ==============================================================================

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp
from scipy.linalg import expm
import cma
import time

# --- Physical Constants (Our Target for Final Analysis) ---
TARGET_RATIO_MUON_ELECTRON = 105.658 / 0.511
TARGET_RATIO_TAU_ELECTRON = 1776.86 / 0.511

# --- Universe Parameters ---
FORGE_UNIVERSE_SIZE = 6
GENES_PER_INTERACTION = 15
NUM_UNIVERSE_GENES = FORGE_UNIVERSE_SIZE * GENES_PER_INTERACTION

# ==============================================================================
#  STEP 1: Build a Universe (Unchanged)
# ==============================================================================

PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']

def build_aoe_from_full_genome(genome: np.ndarray, n_qubits: int) -> np.ndarray:
    if n_qubits < 2: return np.identity(2**n_qubits)
    full_genome = genome.reshape((n_qubits, GENES_PER_INTERACTION))
    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        genes_for_this_interaction = full_genome[i]
        generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=genes_for_this_interaction)
        u_gate_matrix = expm(-1j * generator_h.to_matrix())
        u_gate_op = Operator(u_gate_matrix)
        qc.append(u_gate_op, [i, (i + 1) % n_qubits])
    return Operator(qc).data

# ==============================================================================
#  STEP 2: The NEW Fitness Function - Rewarding Complexity
# ==============================================================================

def get_energy_spectrum(unitary: np.ndarray) -> list:
    eigenvalues = np.linalg.eigvalsh(unitary)
    energies = []
    for eigval in eigenvalues:
        phase = np.angle(eigval)
        energy = np.abs(phase)
        if not (np.isclose(energy, 0.0) or np.isclose(energy, 2 * np.pi)):
            energies.append(energy)
    unique_energies = []
    energies.sort()
    if energies:
        unique_energies.append(energies[0])
        for i in range(1, len(energies)):
            if not np.isclose(energies[i], energies[i-1], rtol=1e-4):
                unique_energies.append(energies[i])
    return unique_energies

def fertility_fitness_function(genome: np.ndarray) -> float:
    """
    A new, simpler fitness function. The AI's only goal is to
    find a universe that is not degenerate.

    The cost is INVERSELY proportional to the number of unique energy levels.
    A simple universe is high-cost. A complex universe is low-cost.
    """
    try:
        U_aoe = build_aoe_from_full_genome(genome, FORGE_UNIVERSE_SIZE)
        energies = get_energy_spectrum(U_aoe)

        num_levels = len(energies)

        # A sterile universe with 0 or 1 level is maximally bad.
        if num_levels < 2:
            return 1000.0

        # The cost is 1 divided by the number of levels.
        # This creates a smooth gradient for the AI to follow.
        # More levels = lower cost.
        cost = 1.0 / num_levels
        return cost

    except Exception:
        return 2000.0

# ==============================================================================
#  STEP 3: Run the AI Fertility Scanner
# ==============================================================================

if __name__ == "__main__":
    print(f"--- The AI Fertility Scanner ---")
    print(f"Objective: Discover the first 'fertile' {FORGE_UNIVERSE_SIZE}-qubit universe")
    print("           by evolving its laws to maximize spectral complexity.")
    print(f"Searching a genetic space of {NUM_UNIVERSE_GENES} parameters.")

    print("\n--- Starting AI Forge to Find a Life-Bearing Universe ---")

    x0 = np.random.uniform(-np.pi, np.pi, NUM_UNIVERSE_GENES)
    sigma0 = 0.5
    options = {'bounds': [-np.pi, np.pi], 'maxfevals': 10000, 'verbose': -9}

    es = cma.CMAEvolutionStrategy(x0, sigma0, options)

    start_time = time.time()
    last_print_time = start_time

    while not es.stop():
        solutions = es.ask()
        fitnesses = [fertility_fitness_function(s) for s in solutions]
        es.tell(solutions, fitnesses)

        current_time = time.time()
        if current_time - last_print_time > 5:
             # The cost is 1/num_levels, so the number of levels is 1/cost
             num_levels_found = 1 / es.result.fbest if es.result.fbest > 0 else "inf"
             print(f"  > Iteration #{es.countiter}, Best Universe Complexity: {num_levels_found:.1f} levels", end='\r')
             last_print_time = current_time

    end_time = time.time()
    print(f"\n  > Forge complete in {end_time - start_time:.2f}s.")

    champion_genome = es.result.xbest

    # ==============================================================================
    #  STEP 4: Analyze the First Fertile Universe
    # ==============================================================================

    print("\n--- Analysis of the First Fertile Universe Discovered ---")

    final_U_aoe = build_aoe_from_full_genome(champion_genome, FORGE_UNIVERSE_SIZE)
    final_energies = get_energy_spectrum(final_U_aoe)

    def analyze_and_report(energies: list):
        if not energies or len(energies) < 3:
            print("The discovered universe is fertile but still too simple for a full lepton comparison.")
            print(f"It has {len(energies)} unique non-zero energy levels.")
            return

        e_electron, e_muon, e_tau = energies[0], energies[1], energies[2]
        predicted_ratio_muon = e_muon / e_electron
        predicted_ratio_tau = e_tau / e_electron

        print("\n--- PREDICTION REPORT FROM THE FIRST FERTILE UNIVERSE ---")
        print(f"This universe has {len(energies)} unique energy levels.")
        print(f"E1 (Electron): {e_electron:.4f} | E2 (Muon): {e_muon:.4f} | E3 (Tau): {e_tau:.4f}")
        print("-" * 50)
        print(f"Predicted Muon/Electron Ratio (E2/E1): {predicted_ratio_muon:.4f}")
        print(f"Actual Muon/Electron Ratio:            {TARGET_RATIO_MUON_ELECTRON:.4f}")
        error_muon = (abs(predicted_ratio_muon - TARGET_RATIO_MUON_ELECTRON) / TARGET_RATIO_MUON_ELECTRON) * 100
        print(f"Error: {error_muon:.2f}%")
        print("-" * 50)
        print(f"Predicted Tau/Electron Ratio (E3/E1): {predicted_ratio_tau:.4f}")
        print(f"Actual Tau/Electron Ratio:            {TARGET_RATIO_TAU_ELECTRON:.4f}")
        error_tau = (abs(predicted_ratio_tau - TARGET_RATIO_TAU_ELECTRON) / TARGET_RATIO_TAU_ELECTRON) * 100
        print(f"Error: {error_tau:.2f}%")

    analyze_and_report(final_energies)

--- The AI Fertility Scanner ---
Objective: Discover the first 'fertile' 6-qubit universe
           by evolving its laws to maximize spectral complexity.
Searching a genetic space of 90 parameters.

--- Starting AI Forge to Find a Life-Bearing Universe ---

  > Forge complete in 1.45s.

--- Analysis of the First Fertile Universe Discovered ---
The discovered universe is fertile but still too simple for a full lepton comparison.
It has 1 unique non-zero energy levels.
